# PN27 — exact-fit child lift

## tl;dr

The frozen one-shot rule `P_hat=N+a+2b+1`, where `a` is the largest exact-fitting wave in
`{1,3,5,9,11,13}` and `b=14-a`, hit primes on **9.010%** of 30,000 fresh odd anchors. The equal-weight matched
offset rate was **8.777%**. A relation-broken offset permutation gave `p=0.0144`, missing the frozen `p<0.01`
strong threshold. Status: **partial predictive support**, not a prime formula.


## Context & Methods

The predictor contains no sieve state, nearby-prime label, prime gap, retry, or fitted parameter. Predictions were
written and SHA-256 frozen before a separate scoring script attached primality labels.

### Key assumptions

- "Fits exactly" means integer divisibility.
- "Largest" means the numerically largest declared wave that divides the anchor.
- Odd anchors are primary because the rule adds an even offset; even anchors are a negative control.
- A hit requires the single frozen candidate itself to be prime.


In [1]:
import csv
import json
from pathlib import Path

HERE = Path.cwd()
results = json.loads((HERE / 'PN27_EXACT_FIT_CHILD_LIFT_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN27_EXACT_FIT_CHILD_LIFT_VALIDATION.json').read_text(encoding='utf-8'))
with (HERE / 'PN27_EXACT_FIT_CHILD_LIFT_VALIDATED_ROWS.csv').open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
print('status:', results['status'])
print('validation:', validation['checks_passed'], '/', validation['checks_total'])
assert results['status'] == 'PARTIAL PREDICTIVE SUPPORT'
assert validation['all_checks_passed'] is True
assert len(rows) == 60000


status: PARTIAL PREDICTIVE SUPPORT
validation: 9 / 9


## Data

In [2]:
print(results['population'])
assert results['population']['odd_primary_rows'] == 30000
assert results['population']['even_control_rows'] == 30000
assert results['population']['protected_87_bit_anchor_used'] is False


{'all_rows': 60000, 'odd_primary_rows': 30000, 'even_control_rows': 30000, 'scales': ['high', 'low', 'middle'], 'protected_87_bit_anchor_used': False}


## Results — worked geometry

In [3]:
N = 35
a = max(w for w in (1,3,5,9,11,13) if N % w == 0)
b = 14 - a
C = a + 2*b
U = N + C
P_hat = U + 1
print({'N': N, 'a': a, 'b': b, 'C': C, 'U': U, 'P_hat': P_hat})
assert (a, b, C, U, P_hat) == (5, 9, 23, 58, 59)
assert results['worked_example_35']['is_prime'] is True


{'N': 35, 'a': 5, 'b': 9, 'C': 23, 'U': 58, 'P_hat': 59}


## Results — fresh one-shot test

In [4]:
headline = results['odd_primary']
print(headline)
print('permutation:', results['offset_permutation_control'])
assert headline['ara_hit_rate'] == 0.0901
assert headline['uniform_allowed_offset_rate'] < headline['ara_hit_rate']
assert results['offset_permutation_control']['one_sided_p_pooled'] >= 0.01


{'n': 30000, 'ara_hit_rate': 0.0901, 'uniform_allowed_offset_rate': 0.08777222222222222, 'difference_vs_uniform': 0.002327777777777786, 'difference_vs_uniform_95ci_normal': [-0.0006352730902790312, 0.005290828645834603], 'fixed_plus_2_rate': 0.08633333333333333, 'difference_vs_fixed_plus_2': 0.003766666666666667}
permutation: {'permutations': 10000, 'seed': 27200, 'alternative': 'ARA hit rate greater than relation-broken offset assignment', 'observed_ara_rate': 0.0901, 'null_mean_rate': 0.08705277666666662, 'null_sd_rate': 0.0013949991083445305, 'one_sided_p_pooled': 0.0143985601439856, 'observed_by_scale': {'high': 0.0768, 'low': 0.11, 'middle': 0.0835}, 'one_sided_p_by_scale': {'high': 0.10168983101689831, 'low': 0.3661633836616338, 'middle': 0.012098790120987902}}


## Results — scale and child-pair detail

In [5]:
print('scale | ARA | uniform | difference')
for scale, values in results['by_scale'].items():
    print(scale, values['ara_hit_rate'], values['uniform_allowed_offset_rate'], values['difference_vs_uniform'])

print()
print('phase A | phase B | n | ARA | uniform | difference')
for group in results['by_child_pair']:
    if group['scale'] == 'pooled':
        print(group['phase_a'], group['phase_b'], group['n'], group['prime_hit_rate'],
              group['uniform_offset_prime_rate'], group['difference_vs_uniform'])


scale | ARA | uniform | difference
high 0.0768 0.07443333333333332 0.002366666666666673
low 0.11 0.10986666666666665 0.000133333333333343
middle 0.0835 0.07901666666666667 0.004483333333333341

phase A | phase B | n | ARA | uniform | difference
13 1 2294 0.09067131647776809 0.07839290903807032 0.01227840743969777
11 3 2569 0.0903075126508369 0.09342156481121058 -0.003114052160373678
9 5 2811 0.13625044468160796 0.09089292066880114 0.04535752401280684
5 9 4510 0.11796008869179601 0.09464153732446415 0.023318551367331866
3 11 4441 0.12362080612474668 0.08774300082563986 0.035877805299106814
1 13 13375 0.05973831775700934 0.08533333333333333 -0.02559501557632398


## Results — even negative control

In [6]:
print(results['even_negative_control'])
assert results['even_negative_control']['all_candidates_even'] is True
assert results['even_negative_control']['prime_hits'] == 0


{'n': 30000, 'prime_hits': 0, 'prime_hit_rate': 0.0, 'all_candidates_even': True}


## Takeaways

1. The exact `35 -> 59` construction is reproduced by the frozen general rule.
2. The rule retained a small positive amount of prime-survival information on fresh odd anchors.
3. The result is suggestive but not decisive: its frozen permutation threshold failed, and the paired 95% interval
   against the equal-weight offset control includes zero.
4. Positive performance is concentrated in the `9↔5`, `5↔9`, and `3↔11` branches. The `1↔13` fallback covers
   almost 45% of anchors and performs poorly.
5. Much of the gain has a direct small-divisor interpretation. PN27 therefore records a useful one-child-layer
   rule, not a new general prime algorithm.
